# Calibrate an objective pair

Each objective looks at a slightly different spot and focuses at a slightly different height. Here you measure that offset by eye, once per objective, against one reference objective; afterwards a change of objective keeps the sample in view.

You need a small, sharp feature (a bead, the corner of a grid). Use ZEN's live view, the joystick and the focus knob as usual; the driver only reads the positions and turns the changer.

In [ ]:
import bootstrap  # noqa: F401 -- puts the repository on the import path

import zmart_drivers.zeiss.light_microscope.zen_api_grpc as zen

# The gateway's control token: copy it from the ZEN API Gateway's tray icon, or leave it
# out when this computer runs the gateway (the driver reads its token file then).
scope = zen.connect(password=None, experiment=None)

from zmart_drivers.zeiss.light_microscope.zen_api_grpc.driver_setup.optical_calibration.calibrate_objective_pair import (
    PairCalibration,
    adopt_calibration,
)

for objective in scope.objectives():
    print(objective["position"], objective["name"])

## The reference objective

Turn to the reference objective (the first pair you measure sets it), centre the feature in the live view and focus on it. Then run the cell.

In [ ]:
pair = PairCalibration(scope.handle)
pair.record_reference()

## The other objective

Write its position below and run the cell: the changer turns, the stage does not move.

In [ ]:
OTHER_POSITION = 2
pair.change_to(OTHER_POSITION)

Now centre and focus the same feature again under this objective, then run the cell.

In [ ]:
pair.record_other()
result = pair.result()
print("offset (um):", [round(v, 2) for v in result["translation_um"]])

## Adopt

When the offset looks right (a few micrometres is usual), save it. Run the notebook again for each further objective.

In [ ]:
saved = adopt_calibration(result)
scope.close()
saved["snapshot"]